[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/11_sliding_window.ipynb)

# 🔴 Hard: Sliding Window Attention

Implement **Sliding Window Attention** — used in Longformer, Mistral, etc. for efficient long-context processing.

Each position $i$ can only attend to positions $j$ where $|i - j| \le w$ (the window size).

### Signature
```python
def sliding_window_attention(Q, K, V, window_size):
    # Q, K, V: (batch, seq, d) → output: (batch, seq, d_v)
    # window_size: int — position i attends to [i-w, i+w]
```

### Rules
- Do **NOT** use sparse attention libraries
- Mask positions outside the window with `-inf`
- `window_size=0`: only self — output should equal V
- `window_size >= seq_len`: equivalent to full attention

In [1]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.5/48.5 kB 1.3 MB/s eta 0:00:00


In [24]:
import torch
import math

In [51]:
# ✏️ YOUR IMPLEMENTATION HERE

def sliding_window_attention(Q, K, V, window_size):
    b,s,d = K.shape
    i = torch.arange(s, device = Q.device).unsqueeze(0)
    j = torch.arange(s, device = Q.device).unsqueeze(1)
    window = (j - i).abs() > window_size
    print(window)
    score = torch.bmm(Q,K.transpose(1,2)) / math.sqrt(d)
    score = score.masked_fill(window, -float('inf'))
    attn_out = torch.softmax(score,dim = -1)@V
    return attn_out

In [52]:
# 🧪 Debug
Q = torch.randn(1, 6, 8)
K = torch.randn(1, 6, 8)
V = torch.randn(1, 6, 8)

out = sliding_window_attention(Q, K, V, window_size=1)
print("Output shape:", out.shape)  # (1, 6, 8)

# window=0 should return V
out0 = sliding_window_attention(Q, K, V, window_size=0)
print("window=0 == V?", torch.allclose(out0, V, atol=1e-5))

tensor([[False, False,  True,  True,  True,  True],
        [False, False, False,  True,  True,  True],
        [ True, False, False, False,  True,  True],
        [ True,  True, False, False, False,  True],
        [ True,  True,  True, False, False, False],
        [ True,  True,  True,  True, False, False]])
Output shape: torch.Size([1, 6, 8])
tensor([[False,  True,  True,  True,  True,  True],
        [ True, False,  True,  True,  True,  True],
        [ True,  True, False,  True,  True,  True],
        [ True,  True,  True, False,  True,  True],
        [ True,  True,  True,  True, False,  True],
        [ True,  True,  True,  True,  True, False]])
window=0 == V? True


In [53]:
from torch_judge import check
check('sliding_window')


🧪 Testing: Sliding Window Attention (Hard)
──────────────────────────────────────────────────
tensor([[False, False, False,  True,  True,  True,  True,  True],
        [False, False, False, False,  True,  True,  True,  True],
        [False, False, False, False, False,  True,  True,  True],
        [ True, False, False, False, False, False,  True,  True],
        [ True,  True, False, False, False, False, False,  True],
        [ True,  True,  True, False, False, False, False, False],
        [ True,  True,  True,  True, False, False, False, False],
        [ True,  True,  True,  True,  True, False, False, False]])
  ✅ [1/5] Output shape (3.0ms)
tensor([[False,  True,  True,  True],
        [ True, False,  True,  True],
        [ True,  True, False,  True],
        [ True,  True,  True, False]])
  ✅ [2/5] window_size=0 — only sees itself (2.6ms)
tensor([[False, False, False, False, False, False],
        [False, False, False, False, False, False],
        [False, False, False, False, 